# Choice多证券与多周期增量验收

本Notebook验证以下完整范围：

1. `provider="choice"`一次请求多只证券；
2. `daily / weekly / monthly`三个周期是否返回不同频率的真实数据；
3. 三只证券的Choice日线是否能写入SQLite；
4. 相同范围连续写入两次后，标准库记录数是否保持不变；
5. Choice直连、SQLite和`provider="qianji", source="choice"`的数据是否一致；
6. Choice周/月线的全空占位记录是否被隔离并留下证券、日期、周期证据；
7. 自动输出Excel和JSON验收证据。

链路：

`Choice直连多周期 → Choice日线适配器 → SQLite → qianji OpenBB Provider → Excel/JSON`

重要边界：当前`daily_bar`表没有`period`字段，只允许写入**不复权日线**。周线、月线只做Choice直连验证，不会写入`daily_bar`，避免不同周期混库。

安全说明：Notebook不会打印或导出Choice账号、密码及`userInfo`令牌。默认使用`ForceLogin=0`，不会主动踢掉其他登录会话。

## 1. 定位项目和检查Python环境

In [2]:
import subprocess
import sys
from pathlib import Path

print("当前 Python：", sys.executable)
print("当前目录：", Path.cwd().resolve())

当前目录 = Path.cwd().resolve()

if 当前目录.name.lower() == "notebooks":
    项目根目录 = 当前目录.parent
else:
    项目根目录 = 当前目录

插件目录 = 项目根目录 / "extensions" / "openbb_choice"
插件配置 = 插件目录 / "pyproject.toml"
修复文件 = 插件目录 / "openbb_choice" / "utils" / "emquant.py"

print("项目根目录：", 项目根目录)
print("插件目录：", 插件目录)
print("插件配置存在：", 插件配置.exists())
print("修复文件存在：", 修复文件.exists())

if not 插件配置.exists():
    raise FileNotFoundError(
        f"没有找到：{插件配置}\n"
        "请确认补丁已经解压到原项目根目录，并覆盖了同名文件。"
    )

# 安装到当前 dm311 环境
安装结果 = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "-e",
        str(插件目录),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("\n安装返回码：", 安装结果.returncode)
print(安装结果.stdout)
print(安装结果.stderr)

if 安装结果.returncode != 0:
    raise RuntimeError("Choice 插件安装失败，请保留上面的完整输出。")

# 重新构建 OpenBB
构建结果 = subprocess.run(
    [
        sys.executable,
        "-c",
        "import openbb; openbb.build()",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("\n构建返回码：", 构建结果.returncode)
print(构建结果.stdout)
print(构建结果.stderr)

if 构建结果.returncode != 0:
    raise RuntimeError("OpenBB 构建失败，请保留上面的完整输出。")

print("\n安装和 OpenBB 构建成功，请彻底重启 Notebook 内核。")

当前 Python： d:\minicoda3\envs\dm311\python.exe
当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
插件目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\extensions\openbb_choice
插件配置存在： True
修复文件存在： True

安装返回码： 0
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Obtaining file:///D:/OneDrive/%E6%A1%8C%E9%9D%A2/%E6%95%B0%E6%8D%AE%E5%9F%BA%E5%BA%A7%E4%BB%A3%E7%A0%81/qianji_openbb_mini/extensions/openbb_choice
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openbb-choic

In [1]:
import os
import sys
from pathlib import Path

print("Python路径：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Conda环境：", os.getenv("CONDA_DEFAULT_ENV", "未检测到"))
print("Notebook当前目录：", Path.cwd().resolve())

# Notebook放在项目notebooks目录时无需修改。
# 若单独存放，请填写项目根目录，例如：
# PROJECT_ROOT_OVERRIDE = r"D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""


def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"指定的项目目录不正确：{candidate}")

    for candidate in (start, *start.parents):
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把Notebook放进项目notebooks文件夹，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("项目根目录：", PROJECT_ROOT)
print(".env是否存在：", (PROJECT_ROOT / ".env").exists())

Python路径： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Conda环境： dm311
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
.env是否存在： True


## 2. 加载安全配置并检查EmQuantAPI、Choice和qianji Provider

In [2]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

from dotenv import load_dotenv

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=True)
else:
    print("提示：未找到.env，将仅使用当前进程已有的环境变量。")


def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "未安装"


try:
    from EmQuantAPI import c as choice_sdk
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError("当前Notebook内核无法导入EmQuantAPI。") from exc

from openbb import obb
from qianji_data_mini import Database
from qianji_data_mini.ingest import ingest_daily

providers = set(obb.coverage.providers)
choice_registered = "choice" in providers
qianji_registered = "qianji" in providers

print("openbb-choice版本：", package_version("openbb-choice"))
print("qianji-data-mini版本：", package_version("qianji-data-mini"))
print("OpenBB发现choice：", choice_registered)
print("OpenBB发现qianji：", qianji_registered)

if not choice_registered or not qianji_registered:
    missing = [name for name, found in [("choice", choice_registered), ("qianji", qianji_registered)] if not found]
    raise RuntimeError(
        f"OpenBB尚未发现Provider：{missing}。请重新运行安装脚本和openbb-build，"
        "然后彻底重启VS Code和Notebook内核。"
    )

choice_package_version = package_version("openbb-choice")
if choice_package_version == "未安装" or Version(choice_package_version) < Version("0.1.2"):
    raise RuntimeError(
        f"当前openbb-choice版本为{choice_package_version}，05号修复版至少需要0.1.2。"
        "请先运行安装Choice_OpenBB插件.bat，确认构建成功，再彻底重启Notebook内核。"
    )

login_mode = os.getenv("CHOICE_LOGIN_MODE", "auto").strip().lower()
username = os.getenv("CHOICE_USERNAME", "").strip()
password = os.getenv("CHOICE_PASSWORD", "")

if login_mode not in {"auto", "userinfo", "password"}:
    raise RuntimeError("CHOICE_LOGIN_MODE只能填写auto、userinfo或password。")
if login_mode == "password" and not (username and password):
    raise RuntimeError("password模式必须同时配置CHOICE_USERNAME和CHOICE_PASSWORD。")

# password模式才把凭据交给OpenBB内存对象；不会打印和导出。
if username:
    obb.user.credentials.choice_username = username
if password:
    obb.user.credentials.choice_password = password

print("Choice登录模式：", login_mode)
print("用户名已配置：", bool(username))
print("密码已配置：", bool(password))

EmQuantAPI：导入成功
openbb-choice版本： 0.1.2
qianji-data-mini版本： 0.3.2
OpenBB发现choice： True
OpenBB发现qianji： True
Choice登录模式： userinfo
用户名已配置： True
密码已配置： True


## 3. 设置多证券、多周期和增量范围

In [3]:
from datetime import date, timedelta

# 正常验收保持True。本地没有Choice环境、只想检查Notebook结构时才改成False。
RUN_REAL_CALLS = True

# False表示即使存在FAIL也先导出证据并显示问题；改成True会在最后主动抛出异常。
STRICT_MODE = False

default_end = date.today() - timedelta(days=1)
default_period_start = default_end - timedelta(days=370)
default_ingest_start = default_end - timedelta(days=45)

raw_symbols = os.getenv(
    "CHOICE_MULTI_VALIDATION_SYMBOLS",
    "000001.SZ,601988.SH,510300.SH",
)
SYMBOLS = list(dict.fromkeys(item.strip().upper() for item in raw_symbols.split(",") if item.strip()))
PERIOD_START_DATE = os.getenv("CHOICE_MULTI_PERIOD_START_DATE", "").strip() or default_period_start.isoformat()
INGEST_START_DATE = os.getenv("CHOICE_MULTI_INGEST_START_DATE", "").strip() or default_ingest_start.isoformat()
END_DATE = os.getenv("CHOICE_MULTI_END_DATE", "").strip() or default_end.isoformat()

if len(SYMBOLS) < 2:
    raise RuntimeError("多证券验收至少需要2只证券，推荐保留默认的3只。")
if date.fromisoformat(PERIOD_START_DATE) > date.fromisoformat(END_DATE):
    raise RuntimeError("CHOICE_MULTI_PERIOD_START_DATE不能晚于结束日期。")
if date.fromisoformat(INGEST_START_DATE) > date.fromisoformat(END_DATE):
    raise RuntimeError("CHOICE_MULTI_INGEST_START_DATE不能晚于结束日期。")

# daily_bar只允许不复权日线。本Notebook强制使用该口径，不受.env中旧值影响。
DAILY_CSD_OPTIONS = "period=1,adjustflag=1,curtype=1,order=1"
os.environ["CHOICE_CSD_OPTIONS"] = DAILY_CSD_OPTIONS

database = Database()
DB_PATH = database.path
OUTPUT_DIR = (PROJECT_ROOT / "validation_output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("真实调用：", RUN_REAL_CALLS)
print("证券列表：", SYMBOLS)
print("多周期对比范围：", PERIOD_START_DATE, "至", END_DATE)
print("增量落库范围：", INGEST_START_DATE, "至", END_DATE)
print("日线落库口径：", DAILY_CSD_OPTIONS)
print("SQLite数据库：", DB_PATH)
print("输出目录：", OUTPUT_DIR)

真实调用： True
证券列表： ['000001.SZ', '601988.SH', '510300.SH']
多周期对比范围： 2025-08-25 至 2026-08-30
增量落库范围： 2026-07-16 至 2026-08-30
日线落库口径： period=1,adjustflag=1,curtype=1,order=1
SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
输出目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output


## 4. 通用的数据整理和脱敏函数

In [4]:
import json
import re
import warnings
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


def safe_error(value: object) -> str:
    text = str(value)
    for secret in [username, password, os.getenv("TUSHARE_TOKEN", "")]:
        if secret:
            text = text.replace(secret, "***")
    return text[:1500]


def normalize_market_frame(frame: pd.DataFrame) -> pd.DataFrame:
    # 把OpenBB/SQLite不同形式整理为同一比较结构。
    if frame is None or frame.empty:
        return pd.DataFrame()

    result = frame.copy()
    if not isinstance(result.index, pd.RangeIndex):
        index_names = {str(name).strip().lower() for name in result.index.names if name is not None}
        column_names = {str(name).strip().lower() for name in result.columns}
        # 某些OpenBB版本同时把date保留为索引和字段，避免reset_index重复插入。
        result = result.reset_index(drop=bool(index_names & column_names))
    result.columns = [str(column).strip().lower() for column in result.columns]

    if "date" not in result.columns:
        for candidate in ["trade_date", "index", "level_0"]:
            if candidate in result.columns:
                result = result.rename(columns={candidate: "date"})
                break
    if "symbol" not in result.columns and len(SYMBOLS) == 1:
        result["symbol"] = SYMBOLS[0]
    if "prev_close" in result.columns and "previous_close" not in result.columns:
        result = result.rename(columns={"prev_close": "previous_close"})

    if "date" in result.columns:
        result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    if "symbol" in result.columns:
        result["symbol"] = result["symbol"].astype(str).str.upper()
    return result


def result_to_frame(result: Any) -> pd.DataFrame:
    return normalize_market_frame(result.to_dataframe())


def count_by_symbol(frame: pd.DataFrame) -> dict[str, int]:
    if frame.empty or "symbol" not in frame.columns:
        return {}
    return {str(key): int(value) for key, value in frame.groupby("symbol").size().items()}


def median_gap_days(frame: pd.DataFrame, symbol: str) -> float | None:
    if frame.empty:
        return None
    dates = pd.to_datetime(
        frame.loc[frame["symbol"] == symbol, "date"].dropna().drop_duplicates().sort_values()
    )
    if len(dates) < 2:
        return None
    return float(dates.diff().dropna().dt.days.median())


print("通用函数加载完成。")

通用函数加载完成。


## 5. Choice直连：一次验证三只证券和三个周期

In [5]:
period_frames: dict[str, pd.DataFrame] = {}
period_calls = []
skipped_placeholder_rows = []

EMPTY_PLACEHOLDER_PATTERN = re.compile(
    r"symbol=(?P<symbol>[^,)]+), date=(?P<date>[^,)]+), "
    r"period=(?P<period>[^,)]+)"
)

for period in ["daily", "weekly", "monthly"]:
    if not RUN_REAL_CALLS:
        period_frames[period] = pd.DataFrame()
        period_calls.append(
            {
                "period": period,
                "status": "SKIP",
                "rows": 0,
                "symbols": 0,
                "skipped_empty_rows": 0,
                "error": "RUN_REAL_CALLS=False",
            }
        )
        continue

    captured_warnings = []
    status = "FAIL"
    rows = 0
    symbols = 0
    error = ""
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")
            result = obb.equity.price.historical(
                symbol=",".join(SYMBOLS),
                start_date=PERIOD_START_DATE,
                end_date=END_DATE,
                period=period,
                use_cache=False,
                provider="choice",
            )
        frame = result_to_frame(result)
        period_frames[period] = frame
        status = "PASS" if not frame.empty else "FAIL"
        rows = len(frame)
        symbols = int(frame["symbol"].nunique()) if "symbol" in frame.columns else 0
    except Exception as exc:
        period_frames[period] = pd.DataFrame()
        error = safe_error(f"{type(exc).__name__}: {exc}")

    current_skipped = []
    for warning_item in captured_warnings:
        message = str(warning_item.message)
        match = EMPTY_PLACEHOLDER_PATTERN.search(message)
        if match:
            row = {
                "symbol": match.group("symbol").strip(),
                "date": match.group("date").strip(),
                "period": match.group("period").strip(),
                "reason": "all_ohlc_empty",
                "action": "skipped_before_openbb_validation",
                "warning_category": warning_item.category.__name__,
            }
            current_skipped.append(row)
            skipped_placeholder_rows.append(row)

    period_calls.append(
        {
            "period": period,
            "status": status,
            "rows": rows,
            "symbols": symbols,
            "skipped_empty_rows": len(current_skipped),
            "error": error,
        }
    )

period_calls_df = pd.DataFrame(period_calls)
skipped_placeholders_df = pd.DataFrame(
    skipped_placeholder_rows,
    columns=["symbol", "date", "period", "reason", "action", "warning_category"],
)
display(period_calls_df)
print("已隔离的全空OHLC占位记录：", len(skipped_placeholders_df))
display(skipped_placeholders_df)

for period, frame in period_frames.items():
    print(period, "各证券行数：", count_by_symbol(frame))
    if not frame.empty:
        display(frame.head(5))

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:44]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:44]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:44]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:49]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:53]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-08-31 23:28:56]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:29:01]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Info][2026-08-31 23:29:06]:percentflag(for csd/css/cses) update success.

[EmQuantAPI Python] [Em_Info][2026-08-31 23:29:17]:heartbeatthread end.

[EmQuantAPI Python] [Em_Info][2026-08-31 23:29:18]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 23:29:18]:verifying your token...

[EmQuantAPI Python] [

,period,status,rows,symbols,skipped_empty_rows,error
0,daily,PASS,738,3,0,
1,weekly,PASS,156,3,0,
2,monthly,PASS,37,3,0,


已隔离的全空OHLC占位记录： 0


,symbol,date,period,reason,action,warning_category


daily 各证券行数： {'000001.SZ': 246, '510300.SH': 246, '601988.SH': 246}


,date,open,high,low,close,volume,symbol,amount,previous_close,change,change_percent,source,volume_unit,amount_unit,timezone
0,2025-08-25,12.120,12.530,12.100,12.450,3.045088e+08,000001.SZ,3.761289e+09,12.060,0.390,0.032338,choice,share,CNY,Asia/Shanghai
1,2025-08-25,5.690,5.700,5.640,5.680,3.163098e+08,601988.SH,1.793267e+09,5.700,-0.020,-0.003509,choice,share,CNY,Asia/Shanghai
2,2025-08-25,4.501,4.572,4.497,4.565,1.456092e+09,510300.SH,6.599627e+09,4.479,0.086,0.019201,choice,share,CNY,Asia/Shanghai
3,2025-08-26,12.450,12.500,12.300,12.360,1.383599e+08,000001.SZ,1.709561e+09,12.450,-0.090,-0.007229,choice,share,CNY,Asia/Shanghai
4,2025-08-26,5.670,5.700,5.650,5.660,2.217984e+08,601988.SH,1.258560e+09,5.680,-0.020,-0.003521,choice,share,CNY,Asia/Shanghai


weekly 各证券行数： {'000001.SZ': 52, '510300.SH': 52, '601988.SH': 52}


,date,open,high,low,close,volume,symbol,amount,previous_close,change,change_percent,source,volume_unit,amount_unit,timezone
0,2025-08-29,12.120,12.530,11.970,12.050,9.747449e+08,000001.SZ,1.191688e+10,12.060,-0.010,-0.000829,choice,share,CNY,Asia/Shanghai
1,2025-08-29,5.690,5.700,5.500,5.520,1.462522e+09,601988.SH,8.195813e+09,5.700,-0.180,-0.031579,choice,share,CNY,Asia/Shanghai
2,2025-08-29,4.501,4.613,4.452,4.601,6.614549e+09,510300.SH,3.003132e+10,4.479,0.122,0.027238,choice,share,CNY,Asia/Shanghai
3,2025-09-05,12.050,12.050,11.600,11.720,6.833369e+08,000001.SZ,8.085144e+09,12.050,-0.330,-0.027386,choice,share,CNY,Asia/Shanghai
4,2025-09-05,5.530,5.650,5.410,5.520,2.143355e+09,601988.SH,1.188626e+10,5.520,0.000,0.000000,choice,share,CNY,Asia/Shanghai


monthly 各证券行数： {'000001.SZ': 12, '510300.SH': 13, '601988.SH': 12}


,date,open,high,low,close,volume,symbol,amount,previous_close,change,change_percent,source,volume_unit,amount_unit,timezone
0,2025-08-29,12.240,12.530,11.940,12.050,2.713280e+09,000001.SZ,3.315677e+10,12.230,-0.180,-0.014718,choice,share,CNY,Asia/Shanghai
1,2025-08-29,5.540,5.740,5.400,5.520,6.994973e+09,601988.SH,3.917143e+10,5.550,-0.030,-0.005405,choice,share,CNY,Asia/Shanghai
2,2025-08-29,4.148,4.613,4.117,4.601,1.890024e+10,510300.SH,8.258341e+10,4.155,0.446,0.107341,choice,share,CNY,Asia/Shanghai
3,2025-09-30,4.607,4.755,4.416,4.741,2.010136e+10,510300.SH,9.253857e+10,4.601,0.140,0.030428,choice,share,CNY,Asia/Shanghai
4,2025-09-30,12.050,12.050,11.270,11.340,2.246812e+09,000001.SZ,2.615262e+10,12.050,-0.710,-0.058921,choice,share,CNY,Asia/Shanghai


## 6. 检查日、周、月频率是否真正不同

In [6]:
frequency_rows = []
expected_gap = {
    "daily": (1, 5),
    "weekly": (5, 10),
    "monthly": (20, 40),
}

for period in ["daily", "weekly", "monthly"]:
    frame = period_frames.get(period, pd.DataFrame())
    lower, upper = expected_gap[period]
    for symbol in SYMBOLS:
        subset = frame[frame["symbol"] == symbol] if not frame.empty and "symbol" in frame.columns else pd.DataFrame()
        gap = median_gap_days(frame, symbol) if not frame.empty else None
        frequency_rows.append(
            {
                "period": period,
                "symbol": symbol,
                "rows": len(subset),
                "first_date": str(subset["date"].min()) if not subset.empty else "",
                "last_date": str(subset["date"].max()) if not subset.empty else "",
                "median_calendar_gap_days": gap,
                "expected_gap_days": f"{lower}-{upper}",
                "frequency_check": (
                    "PASS" if gap is not None and lower <= gap <= upper else "FAIL"
                ),
            }
        )

frequency_df = pd.DataFrame(frequency_rows)

ordering_rows = []
for symbol in SYMBOLS:
    counts = {
        period: int((frame["symbol"] == symbol).sum())
        if not frame.empty and "symbol" in frame.columns
        else 0
        for period, frame in period_frames.items()
    }
    ordering_rows.append(
        {
            "symbol": symbol,
            "daily_rows": counts.get("daily", 0),
            "weekly_rows": counts.get("weekly", 0),
            "monthly_rows": counts.get("monthly", 0),
            "daily_gt_weekly_gt_monthly": (
                counts.get("daily", 0) > counts.get("weekly", 0) > counts.get("monthly", 0) > 0
            ),
        }
    )

period_ordering_df = pd.DataFrame(ordering_rows)
display(frequency_df)
display(period_ordering_df)

,period,symbol,rows,first_date,last_date,median_calendar_gap_days,expected_gap_days,frequency_check
0,daily,000001.SZ,246,2025-08-25,2026-08-28,1.0,1-5,PASS
1,daily,601988.SH,246,2025-08-25,2026-08-28,1.0,1-5,PASS
2,daily,510300.SH,246,2025-08-25,2026-08-28,1.0,1-5,PASS
3,weekly,000001.SZ,52,2025-08-29,2026-08-28,7.0,5-10,PASS
4,weekly,601988.SH,52,2025-08-29,2026-08-28,7.0,5-10,PASS
5,weekly,510300.SH,52,2025-08-29,2026-08-28,7.0,5-10,PASS
6,monthly,000001.SZ,12,2025-08-29,2026-07-31,31.0,20-40,PASS
7,monthly,601988.SH,12,2025-08-29,2026-07-31,31.0,20-40,PASS
8,monthly,510300.SH,13,2025-08-29,2026-08-28,30.5,20-40,PASS


,symbol,daily_rows,weekly_rows,monthly_rows,daily_gt_weekly_gt_monthly
0,000001.SZ,246,52,12,True
1,601988.SH,246,52,12,True
2,510300.SH,246,52,13,True


## 7. Choice日线增量落库并重复运行两次

这里强制使用`period=1, adjustflag=1`。`stored_rows`表示本次写入或更新数量；真正的幂等验收看限定范围内的数据库唯一记录数是否保持不变。

In [7]:
def load_db_daily() -> pd.DataFrame:
    frames = []
    for symbol in SYMBOLS:
        frame = database.query_dataframe(
            symbol=symbol,
            source="choice",
            start_date=INGEST_START_DATE,
            end_date=END_DATE,
        )
        if not frame.empty:
            frames.append(frame)
    return normalize_market_frame(pd.concat(frames, ignore_index=True)) if frames else pd.DataFrame()


db_before_df = load_db_daily()
count_before = len(db_before_df)
first_ingest = None
second_ingest = None
first_ingest_error = ""
second_ingest_error = ""

if RUN_REAL_CALLS:
    try:
        first_ingest = ingest_daily(
            source="choice",
            symbols=SYMBOLS,
            start_date=INGEST_START_DATE,
            end_date=END_DATE,
            database_path=DB_PATH,
        )
    except Exception as exc:
        first_ingest_error = safe_error(f"{type(exc).__name__}: {exc}")

db_after_first_df = load_db_daily()
count_after_first = len(db_after_first_df)

if RUN_REAL_CALLS and not first_ingest_error:
    try:
        second_ingest = ingest_daily(
            source="choice",
            symbols=SYMBOLS,
            start_date=INGEST_START_DATE,
            end_date=END_DATE,
            database_path=DB_PATH,
        )
    except Exception as exc:
        second_ingest_error = safe_error(f"{type(exc).__name__}: {exc}")

db_after_second_df = load_db_daily()
count_after_second = len(db_after_second_df)
idempotent = count_after_first == count_after_second and count_after_first > 0


def ingest_record(label: str, result, error: str) -> dict:
    if result is None:
        return {
            "run": label,
            "status": "SKIP" if not RUN_REAL_CALLS else "FAIL",
            "received_rows": 0,
            "stored_rows": 0,
            "failed_symbols": "",
            "error": error or ("RUN_REAL_CALLS=False" if not RUN_REAL_CALLS else "未执行"),
        }
    failed = {key: safe_error(value) for key, value in result.failed_symbols.items()}
    return {
        "run": label,
        "status": "PASS" if not failed else "PARTIAL",
        "received_rows": result.received_rows,
        "stored_rows": result.stored_rows,
        "failed_symbols": json.dumps(failed, ensure_ascii=False),
        "error": error,
    }


ingestion_runs_df = pd.DataFrame(
    [
        ingest_record("第一次", first_ingest, first_ingest_error),
        ingest_record("第二次", second_ingest, second_ingest_error),
    ]
)
idempotency_df = pd.DataFrame(
    [
        {
            "count_before": count_before,
            "count_after_first": count_after_first,
            "count_after_second": count_after_second,
            "second_minus_first": count_after_second - count_after_first,
            "idempotent": idempotent,
        }
    ]
)

display(ingestion_runs_df)
display(idempotency_df)
print("数据库各证券行数：", count_by_symbol(db_after_second_df))
display(db_after_second_df.head(10))

[EmQuantAPI Python] [Em_Info][2026-08-31 23:30:48]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 23:30:48]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:30:48]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:30:50]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 23:30:54]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:06]:heartbeatthread end.

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:11]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:11]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:11]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:14]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:18]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 23:31:25]:heartbeatthread end.



,run,status,received_rows,stored_rows,failed_symbols,error
0,第一次,PASS,96,96,{},
1,第二次,PASS,96,96,{},


,count_before,count_after_first,count_after_second,second_minus_first,idempotent
0,96,96,96,0,True


数据库各证券行数： {'000001.SZ': 32, '510300.SH': 32, '601988.SH': 32}


,source,symbol,adjustment,open,high,low,close,volume,amount,previous_close,change_percent,currency,timezone,volume_unit,amount_unit,ingested_at,date
0,choice,000001.SZ,unadjusted,10.85,10.93,10.72,10.77,80076623.0,8.644359e+08,10.84,-0.6458,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-16
1,choice,000001.SZ,unadjusted,10.75,10.88,10.72,10.78,107549901.0,1.163189e+09,10.77,0.0929,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-17
2,choice,000001.SZ,unadjusted,10.75,11.00,10.74,10.98,156730393.0,1.713460e+09,10.78,1.8553,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-20
3,choice,000001.SZ,unadjusted,10.99,11.13,10.83,10.84,175511288.0,1.925299e+09,10.98,-1.2750,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-21
4,choice,000001.SZ,unadjusted,10.81,10.98,10.77,10.98,102948394.0,1.120151e+09,10.84,1.2915,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-22
5,choice,000001.SZ,unadjusted,10.92,11.12,10.90,11.08,109574268.0,1.210838e+09,10.98,0.9107,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-23
6,choice,000001.SZ,unadjusted,11.09,11.18,11.09,11.10,114093292.0,1.269361e+09,11.08,0.1805,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-24
7,choice,000001.SZ,unadjusted,11.11,11.16,11.04,11.11,95715556.0,1.062796e+09,11.10,0.0901,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-27
8,choice,000001.SZ,unadjusted,11.10,11.21,11.09,11.20,106101129.0,1.185515e+09,11.11,0.8101,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-28
9,choice,000001.SZ,unadjusted,11.19,11.36,11.18,11.28,151105407.0,1.705864e+09,11.20,0.7143,CNY,Asia/Shanghai,share,CNY,2026-09-01T06:31:21.218280+00:00,2026-07-29


## 8. 检查SQLite日线质量

In [8]:
db_daily_df = db_after_second_df.copy()

if not db_daily_df.empty:
    key_fields = ["source", "symbol", "date", "adjustment"]
    required_fields = ["source", "symbol", "date", "open", "high", "low", "close"]
    duplicate_keys = int(db_daily_df.duplicated(subset=key_fields).sum())
    missing_required = int(db_daily_df[required_fields].isna().any(axis=1).sum())

    ohlc = db_daily_df[["open", "high", "low", "close"]].apply(pd.to_numeric, errors="coerce")
    complete = ohlc.notna().all(axis=1)
    ohlc_errors = int(
        (
            complete
            & (
                (ohlc["high"] < ohlc.max(axis=1))
                | (ohlc["low"] > ohlc.min(axis=1))
            )
        ).sum()
    )
    negative_volume = int((pd.to_numeric(db_daily_df["volume"], errors="coerce").dropna() < 0).sum())
    unit_ok = (
        set(db_daily_df["volume_unit"].dropna().astype(str)) <= {"share"}
        and set(db_daily_df["amount_unit"].dropna().astype(str)) <= {"CNY"}
    )
else:
    duplicate_keys = missing_required = ohlc_errors = negative_volume = None
    unit_ok = None

with database.connect() as connection:
    db_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])

db_quality_df = pd.DataFrame(
    [
        ["数据库记录非空", ">0", len(db_daily_df), len(db_daily_df) > 0],
        ["覆盖全部证券", f"={len(SYMBOLS)}", int(db_daily_df["symbol"].nunique()) if not db_daily_df.empty else 0, not db_daily_df.empty and set(SYMBOLS) <= set(db_daily_df["symbol"])],
        ["主键重复", "=0", duplicate_keys if duplicate_keys is not None else "未检查", duplicate_keys == 0 if duplicate_keys is not None else False],
        ["关键字段缺失", "=0", missing_required if missing_required is not None else "未检查", missing_required == 0 if missing_required is not None else False],
        ["OHLC异常", "=0", ohlc_errors if ohlc_errors is not None else "未检查", ohlc_errors == 0 if ohlc_errors is not None else False],
        ["成交量负数", "=0", negative_volume if negative_volume is not None else "未检查", negative_volume == 0 if negative_volume is not None else False],
        ["标准单位", "share/CNY", "符合" if unit_ok else "不符合或未检查", unit_ok is True],
        ["SQLite完整性", "ok", db_integrity, db_integrity == "ok"],
    ],
    columns=["check", "threshold", "actual", "passed"],
)
display(db_quality_df)

,check,threshold,actual,passed
0,数据库记录非空,>0,96,True
1,覆盖全部证券,=3,3,True
2,主键重复,=0,0,True
3,关键字段缺失,=0,0,True
4,OHLC异常,=0,0,True
5,成交量负数,=0,0,True
6,标准单位,share/CNY,符合,True
7,SQLite完整性,ok,ok,True


## 9. 使用qianji Provider逐只证券读取SQLite

In [9]:
qianji_frames = []
qianji_read_rows = []

for symbol in SYMBOLS:
    try:
        result = obb.equity.price.historical(
            symbol=symbol,
            start_date=INGEST_START_DATE,
            end_date=END_DATE,
            source="choice",
            provider="qianji",
        )
        frame = result_to_frame(result)
        qianji_frames.append(frame)
        qianji_read_rows.append(
            {"symbol": symbol, "status": "PASS", "rows": len(frame), "error": ""}
        )
    except Exception as exc:
        qianji_read_rows.append(
            {"symbol": symbol, "status": "FAIL", "rows": 0, "error": safe_error(f"{type(exc).__name__}: {exc}")}
        )

qianji_df = (
    normalize_market_frame(pd.concat(qianji_frames, ignore_index=True))
    if qianji_frames
    else pd.DataFrame()
)
qianji_reads_df = pd.DataFrame(qianji_read_rows)

display(qianji_reads_df)
print("qianji总行数：", len(qianji_df))
display(qianji_df.head(10))

,symbol,status,rows,error
0,000001.SZ,PASS,32,
1,601988.SH,PASS,32,
2,510300.SH,PASS,32,


qianji总行数： 96


,date,open,high,low,close,volume,source,amount,previous_close,change_percent,currency,timezone,volume_unit,amount_unit,symbol,adjustment,ingested_at
0,2026-07-16,10.85,10.93,10.72,10.77,80076623.0,choice,8.644359e+08,10.84,-0.006458,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
1,2026-07-17,10.75,10.88,10.72,10.78,107549901.0,choice,1.163189e+09,10.77,0.000929,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
2,2026-07-20,10.75,11.00,10.74,10.98,156730393.0,choice,1.713460e+09,10.78,0.018553,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
3,2026-07-21,10.99,11.13,10.83,10.84,175511288.0,choice,1.925299e+09,10.98,-0.012750,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
4,2026-07-22,10.81,10.98,10.77,10.98,102948394.0,choice,1.120151e+09,10.84,0.012915,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
5,2026-07-23,10.92,11.12,10.90,11.08,109574268.0,choice,1.210838e+09,10.98,0.009107,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
6,2026-07-24,11.09,11.18,11.09,11.10,114093292.0,choice,1.269361e+09,11.08,0.001805,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
7,2026-07-27,11.11,11.16,11.04,11.11,95715556.0,choice,1.062796e+09,11.10,0.000901,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
8,2026-07-28,11.10,11.21,11.09,11.20,106101129.0,choice,1.185515e+09,11.11,0.008101,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00
9,2026-07-29,11.19,11.36,11.18,11.28,151105407.0,choice,1.705864e+09,11.20,0.007143,CNY,Asia/Shanghai,share,CNY,000001.SZ,unadjusted,2026-09-01T06:31:21.218280+00:00


## 10. 比较Choice直连、SQLite和qianji结果

In [10]:
COMPARE_FIELDS = [
    "open", "high", "low", "close", "volume", "amount",
    "previous_close", "change_percent",
]


def comparison_ready(frame: pd.DataFrame, *, sqlite_layer: bool = False) -> pd.DataFrame:
    result = normalize_market_frame(frame)
    if result.empty:
        return result
    if sqlite_layer and "change_percent" in result.columns:
        # SQLite保存百分点，OpenBB消费层保存小数。
        result["change_percent"] = pd.to_numeric(result["change_percent"], errors="coerce") / 100
    columns = [column for column in ["symbol", "date", *COMPARE_FIELDS] if column in result.columns]
    return result[columns].copy()


def compare_layers(left: pd.DataFrame, right: pd.DataFrame, left_name: str, right_name: str):
    summary_rows = []
    detail = pd.DataFrame()
    if left.empty or right.empty:
        for field in COMPARE_FIELDS:
            summary_rows.append(
                {
                    "left": left_name,
                    "right": right_name,
                    "field": field,
                    "matched_rows": 0,
                    "max_abs_diff": None,
                    "mismatch_rows": None,
                    "status": "SKIP",
                }
            )
        return pd.DataFrame(summary_rows), detail

    common_fields = [field for field in COMPARE_FIELDS if field in left.columns and field in right.columns]
    detail = left[["symbol", "date", *common_fields]].merge(
        right[["symbol", "date", *common_fields]],
        on=["symbol", "date"],
        suffixes=(f"_{left_name}", f"_{right_name}"),
        how="inner",
    )

    for field in COMPARE_FIELDS:
        left_column = f"{field}_{left_name}"
        right_column = f"{field}_{right_name}"
        if left_column not in detail.columns or right_column not in detail.columns:
            summary_rows.append(
                {
                    "left": left_name,
                    "right": right_name,
                    "field": field,
                    "matched_rows": len(detail),
                    "max_abs_diff": None,
                    "mismatch_rows": None,
                    "status": "SKIP",
                }
            )
            continue

        left_values = pd.to_numeric(detail[left_column], errors="coerce")
        right_values = pd.to_numeric(detail[right_column], errors="coerce")
        both_missing = left_values.isna() & right_values.isna()
        close_enough = np.isclose(left_values, right_values, rtol=1e-9, atol=1e-6, equal_nan=True)
        mismatches = int((~close_enough & ~both_missing).sum())
        differences = (left_values - right_values).abs().dropna()
        summary_rows.append(
            {
                "left": left_name,
                "right": right_name,
                "field": field,
                "matched_rows": len(detail),
                "max_abs_diff": float(differences.max()) if not differences.empty else 0.0,
                "mismatch_rows": mismatches,
                "status": "PASS" if mismatches == 0 and len(detail) > 0 else "FAIL",
            }
        )
    return pd.DataFrame(summary_rows), detail


direct_daily_df = period_frames.get("daily", pd.DataFrame())
if not direct_daily_df.empty:
    direct_daily_df = direct_daily_df[
        (direct_daily_df["date"] >= INGEST_START_DATE)
        & (direct_daily_df["date"] <= END_DATE)
    ].copy()

direct_ready = comparison_ready(direct_daily_df)
db_ready = comparison_ready(db_daily_df, sqlite_layer=True)
qianji_ready = comparison_ready(qianji_df)

direct_db_summary_df, direct_db_detail_df = compare_layers(
    direct_ready, db_ready, "choice", "sqlite"
)
direct_qianji_summary_df, direct_qianji_detail_df = compare_layers(
    direct_ready, qianji_ready, "choice", "qianji"
)

comparison_summary_df = pd.concat(
    [direct_db_summary_df, direct_qianji_summary_df], ignore_index=True
)
display(comparison_summary_df)

,left,right,field,matched_rows,max_abs_diff,mismatch_rows,status
0,choice,sqlite,open,96,0.0,0,PASS
1,choice,sqlite,high,96,0.0,0,PASS
2,choice,sqlite,low,96,0.0,0,PASS
3,choice,sqlite,close,96,0.0,0,PASS
4,choice,sqlite,volume,96,0.0,0,PASS
5,choice,sqlite,amount,96,0.0,0,PASS
6,choice,sqlite,previous_close,96,0.0,0,PASS
7,choice,sqlite,change_percent,96,0.0,0,PASS
8,choice,qianji,open,96,0.0,0,PASS
9,choice,qianji,high,96,0.0,0,PASS


## 11. 汇总硬性验收指标

In [11]:
quality_rows = []


def add_gate(category: str, check: str, threshold: str, actual: object, passed: bool, evidence: str):
    quality_rows.append(
        {
            "category": category,
            "check": check,
            "threshold": threshold,
            "actual": actual,
            "status": "PASS" if passed else "FAIL",
            "evidence": evidence,
        }
    )


call_status = dict(zip(period_calls_df["period"], period_calls_df["status"]))
period_symbol_counts = {
    period: set(frame["symbol"].unique()) if not frame.empty and "symbol" in frame.columns else set()
    for period, frame in period_frames.items()
}

for period in ["daily", "weekly", "monthly"]:
    add_gate(
        "多周期",
        f"{period}直连成功",
        "PASS",
        call_status.get(period, "未执行"),
        call_status.get(period) == "PASS",
        "period_calls",
    )
    add_gate(
        "多证券",
        f"{period}覆盖全部证券",
        f"{len(SYMBOLS)}只",
        len(period_symbol_counts.get(period, set())),
        set(SYMBOLS) <= period_symbol_counts.get(period, set()),
        "frequency_detail",
    )

add_gate(
    "多周期",
    "日周月中位间隔符合预期",
    "全部PASS",
    int((frequency_df["frequency_check"] == "PASS").sum()),
    not frequency_df.empty and (frequency_df["frequency_check"] == "PASS").all(),
    "frequency_detail",
)
add_gate(
    "多周期",
    "日线行数>周线>月线",
    "全部证券为True",
    int(period_ordering_df["daily_gt_weekly_gt_monthly"].sum()),
    not period_ordering_df.empty and period_ordering_df["daily_gt_weekly_gt_monthly"].all(),
    "period_ordering",
)

standard_ohlc_missing = 0
for frame in period_frames.values():
    if not frame.empty and set(["open", "high", "low", "close"]) <= set(frame.columns):
        standard_ohlc_missing += int(frame[["open", "high", "low", "close"]].isna().any(axis=1).sum())
add_gate(
    "源数据占位",
    "全空占位未进入OpenBB标准结果",
    "标准结果OHLC缺失数=0",
    f"缺失={standard_ohlc_missing}；隔离={len(skipped_placeholders_df)}",
    standard_ohlc_missing == 0 and all(status == "PASS" for status in call_status.values()),
    "empty_placeholders",
)

first_failed = first_ingest.failed_symbols if first_ingest is not None else {"run": first_ingest_error or "未执行"}
second_failed = second_ingest.failed_symbols if second_ingest is not None else {"run": second_ingest_error or "未执行"}
add_gate("增量落库", "第一次落库无失败证券", "失败数=0", len(first_failed), len(first_failed) == 0, "ingestion_runs")
add_gate("增量落库", "第二次落库无失败证券", "失败数=0", len(second_failed), len(second_failed) == 0, "ingestion_runs")
add_gate("增量落库", "重复运行幂等", "第二次后记录数=第一次后记录数", idempotent, idempotent, "idempotency")

for row in db_quality_df.to_dict(orient="records"):
    add_gate("SQLite质量", row["check"], str(row["threshold"]), row["actual"], bool(row["passed"]), "db_quality")

qianji_all_pass = not qianji_reads_df.empty and (qianji_reads_df["status"] == "PASS").all()
add_gate("OpenBB读库", "qianji逐只证券读取成功", "全部PASS", int((qianji_reads_df["status"] == "PASS").sum()), qianji_all_pass, "qianji_reads")

direct_db_pass = (
    not direct_db_summary_df.empty
    and (direct_db_summary_df["status"] == "PASS").all()
    and len(direct_db_detail_df) == len(db_ready) == len(direct_ready)
)
direct_qianji_pass = (
    not direct_qianji_summary_df.empty
    and (direct_qianji_summary_df["status"] == "PASS").all()
    and len(direct_qianji_detail_df) == len(qianji_ready) == len(direct_ready)
)
add_gate("跨层一致性", "Choice直连=SQLite", "字段差异0且行数一致", len(direct_db_detail_df), direct_db_pass, "comparison_summary")
add_gate("跨层一致性", "Choice直连=qianji", "字段差异0且行数一致", len(direct_qianji_detail_df), direct_qianji_pass, "comparison_summary")

quality_gates_df = pd.DataFrame(quality_rows)
passed_gates = int((quality_gates_df["status"] == "PASS").sum())
failed_gates = int((quality_gates_df["status"] == "FAIL").sum())

print("PASS数量：", passed_gates)
print("FAIL数量：", failed_gates)
display(quality_gates_df)

PASS数量： 23
FAIL数量： 0


,category,check,threshold,actual,status,evidence
0,多周期,daily直连成功,PASS,PASS,PASS,period_calls
1,多证券,daily覆盖全部证券,3只,3,PASS,frequency_detail
2,多周期,weekly直连成功,PASS,PASS,PASS,period_calls
3,多证券,weekly覆盖全部证券,3只,3,PASS,frequency_detail
4,多周期,monthly直连成功,PASS,PASS,PASS,period_calls
5,多证券,monthly覆盖全部证券,3只,3,PASS,frequency_detail
6,多周期,日周月中位间隔符合预期,全部PASS,9,PASS,frequency_detail
7,多周期,日线行数>周线>月线,全部证券为True,3,PASS,period_ordering
8,源数据占位,全空占位未进入OpenBB标准结果,标准结果OHLC缺失数=0,缺失=0；隔离=0,PASS,empty_placeholders
9,增量落库,第一次落库无失败证券,失败数=0,0,PASS,ingestion_runs


## 12. 导出Excel和JSON验收证据

In [12]:
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice多证券多周期增量验收_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice多证券多周期增量验收_{timestamp}.json"

overview_df = pd.DataFrame(
    [
        ["generated_at", pd.Timestamp.now(tz="UTC").isoformat()],
        ["python", sys.executable],
        ["project_root", str(PROJECT_ROOT)],
        ["database", str(DB_PATH)],
        ["symbols", ",".join(SYMBOLS)],
        ["period_range", f"{PERIOD_START_DATE} 至 {END_DATE}"],
        ["ingest_range", f"{INGEST_START_DATE} 至 {END_DATE}"],
        ["daily_options", DAILY_CSD_OPTIONS],
        ["openbb_choice_version", package_version("openbb-choice")],
        ["qianji_data_mini_version", package_version("qianji-data-mini")],
        ["passed_gates", passed_gates],
        ["failed_gates", failed_gates],
        ["skipped_empty_placeholders", len(skipped_placeholders_df)],
        ["credentials_included", False],
        ["boundary", "周线/月线只做Choice直连验证；SQLite仅写入不复权日线"],
    ],
    columns=["item", "value"],
)

sheet_frames = {
    "验收概览": overview_df,
    "质量门槛": quality_gates_df,
    "周期调用": period_calls_df,
    "月线空占位": skipped_placeholders_df,
    "频率明细": frequency_df,
    "周期行数": period_ordering_df,
    "落库执行": ingestion_runs_df,
    "幂等检查": idempotency_df,
    "数据库质量": db_quality_df,
    "qianji读取": qianji_reads_df,
    "跨层对比": comparison_summary_df,
    "Choice日线样本": direct_daily_df.head(500),
    "Choice周线样本": period_frames.get("weekly", pd.DataFrame()).head(500),
    "Choice月线样本": period_frames.get("monthly", pd.DataFrame()).head(500),
    "SQLite日线": db_daily_df.head(1000),
    "qianji日线": qianji_df.head(1000),
}


def safe_cell(value):
    if isinstance(value, str) and value[:1] in {"=", "+", "-", "@"}:
        return "'" + value
    return value


safe_frames = {}
for sheet_name, frame in sheet_frames.items():
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        safe_frame[column] = safe_frame[column].map(safe_cell)
    safe_frames[sheet_name] = safe_frame

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in safe_frames.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False, startrow=3)

workbook = load_workbook(excel_path)
NAVY, BLUE, LIGHT_BLUE = "17365D", "2F75B5", "D9EAF7"
GREEN, YELLOW, RED, WHITE, GRID = "E2F0D9", "FFF2CC", "FCE4D6", "FFFFFF", "B7C9D6"
thin = Side(style="thin", color=GRID)

for worksheet in workbook.worksheets:
    frame = safe_frames[worksheet.title]
    max_col = max(1, len(frame.columns))
    max_row = worksheet.max_row
    last_col = get_column_letter(max_col)

    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max_col)
    title = worksheet.cell(1, 1, f"Choice多证券多周期增量验收｜{worksheet.title}")
    title.fill = PatternFill("solid", fgColor=NAVY)
    title.font = Font(name="Microsoft YaHei", size=15, bold=True, color=WHITE)
    title.alignment = Alignment(vertical="center")
    worksheet.row_dimensions[1].height = 28

    worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max_col)
    subtitle = worksheet.cell(2, 1, "真实结果与自动验收证据；凭据未导出")
    subtitle.fill = PatternFill("solid", fgColor=LIGHT_BLUE)
    subtitle.font = Font(name="Microsoft YaHei", size=10, color=NAVY)

    for cell in worksheet[4]:
        cell.fill = PatternFill("solid", fgColor=BLUE)
        cell.font = Font(name="Microsoft YaHei", bold=True, color=WHITE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
    worksheet.row_dimensions[4].height = 34

    for row in worksheet.iter_rows(min_row=5, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.font = Font(name="Microsoft YaHei", size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
            text = str(cell.value or "")
            if text == "PASS" or text == "True":
                cell.fill = PatternFill("solid", fgColor=GREEN)
            elif text.startswith("FAIL") or text == "False":
                cell.fill = PatternFill("solid", fgColor=RED)
            elif text in {"SKIP", "PARTIAL"}:
                cell.fill = PatternFill("solid", fgColor=YELLOW)

    for column_index, column_name in enumerate(frame.columns, start=1):
        values = [str(column_name)] + [str(value or "") for value in frame[column_name].head(200)]
        longest = max((len(value) for value in values), default=8)
        worksheet.column_dimensions[get_column_letter(column_index)].width = min(max(longest * 1.1 + 2, 10), 40)

    worksheet.freeze_panes = "A5"
    worksheet.auto_filter.ref = f"A4:{last_col}{max_row}"
    worksheet.sheet_view.showGridLines = False
    worksheet.print_title_rows = "1:4"
    worksheet.page_setup.orientation = "landscape"
    worksheet.page_setup.fitToWidth = 1
    worksheet.sheet_properties.pageSetUpPr.fitToPage = True

workbook.save(excel_path)

json_payload = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "python": sys.executable,
    "project_root": str(PROJECT_ROOT),
    "database": str(DB_PATH),
    "symbols": SYMBOLS,
    "period_range": [PERIOD_START_DATE, END_DATE],
    "ingest_range": [INGEST_START_DATE, END_DATE],
    "credentials_included": False,
    "boundary": "weekly/monthly direct validation only; SQLite stores unadjusted daily bars",
    "period_calls": period_calls_df.to_dict(orient="records"),
    "skipped_empty_placeholders": skipped_placeholders_df.to_dict(orient="records"),
    "frequency_checks": frequency_df.where(pd.notna(frequency_df), None).to_dict(orient="records"),
    "period_ordering": period_ordering_df.to_dict(orient="records"),
    "ingestion_runs": ingestion_runs_df.to_dict(orient="records"),
    "idempotency": idempotency_df.to_dict(orient="records"),
    "database_quality": db_quality_df.to_dict(orient="records"),
    "qianji_reads": qianji_reads_df.to_dict(orient="records"),
    "comparison_summary": comparison_summary_df.where(pd.notna(comparison_summary_df), None).to_dict(orient="records"),
    "quality_gates": quality_gates_df.to_dict(orient="records"),
    "passed_gates": passed_gates,
    "failed_gates": failed_gates,
}
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Excel已生成：", excel_path)
print("JSON已生成：", json_path)
print("Excel大小（字节）：", excel_path.stat().st_size)
print("JSON大小（字节）：", json_path.stat().st_size)

Excel已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice多证券多周期增量验收_20260831_233216.xlsx
JSON已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice多证券多周期增量验收_20260831_233216.json
Excel大小（字节）： 68596
JSON大小（字节）： 14107


## 13. 自动复核输出并给出最终结论

In [13]:
from openpyxl import load_workbook

expected_sheets = list(safe_frames)
check_workbook = load_workbook(excel_path, read_only=True, data_only=False)
actual_sheets = check_workbook.sheetnames
missing_sheets = sorted(set(expected_sheets) - set(actual_sheets))

json_check = json.loads(json_path.read_text(encoding="utf-8"))
export_checks_df = pd.DataFrame(
    [
        ["Excel文件存在", excel_path.exists(), str(excel_path)],
        ["JSON文件存在", json_path.exists(), str(json_path)],
        ["工作表完整", not missing_sheets, f"缺少：{missing_sheets}" if missing_sheets else f"共{len(actual_sheets)}张表"],
        ["凭据未导出", json_check.get("credentials_included") is False, "credentials_included=False"],
        ["质量门槛已生成", len(quality_gates_df) > 0, f"PASS={passed_gates}, FAIL={failed_gates}"],
    ],
    columns=["check", "passed", "detail"],
)
display(export_checks_df)

if not export_checks_df["passed"].all():
    raise RuntimeError("导出文件复核失败，请查看上表。")

failed_checks = quality_gates_df.loc[quality_gates_df["status"] == "FAIL", ["category", "check", "actual"]]
if failed_checks.empty:
    print("最终结论：全部硬性验收指标通过。")
else:
    print("最终结论：证据已导出，但仍有未通过项目：")
    display(failed_checks)
    if STRICT_MODE:
        raise RuntimeError(f"存在{len(failed_checks)}项硬性验收失败。")

,check,passed,detail
0,Excel文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
1,JSON文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
2,工作表完整,True,共16张表
3,凭据未导出,True,credentials_included=False
4,质量门槛已生成,True,"PASS=23, FAIL=0"


最终结论：全部硬性验收指标通过。
